# **Classification: Stacking Classifier**

## **Justification of Preprocessing Strategy**

### **The Scale Sensitivity of a Heterogeneous Stack**
The **Stacking Classifier** combines multiple "Champion" models identified in previous experiments. This ensemble is inherently heterogeneous, featuring distance-based models (**KNN, Logistic Regression**) alongside scale-invariant models (**Decision Trees, Naive Bayes**). While the base models are already optimized, the way they interact and the performance of the meta-model (Logistic Regression) can vary significantly depending on the feature distribution. Therefore, we will test both **Standardization** and **Normalization** to determine which scaling method leads to a more harmonious integration of these diverse specialists.

### **Leveraging Pre-Optimized Experts**
Following the strategy of reusing "Champion" parameters, we eliminate the need for further hyperparameter optimization at this stage. We treat each base model as a fixed "expert" using the exact parameters that yielded the best **Recall** in previous runs. The meta-model (Level 1) will learn to weigh the predictions of these experts. Since the meta-model itself is a Logistic Regression, it also benefits from scaled inputs to ensure stable and efficient convergence when processing the outputs of the Level 0 models.

---

## **Experiment Design**

We have defined a tournament of **2 focused experiments** to identify the best environment for our optimized ensemble:

* **Standardized Champion Stack**: A Stacking ensemble using our **Optimized Base Models** (Champions) trained and evaluated on data scaled via `StandardScaler`.
* **Normalized Champion Stack**: The same ensemble of **Optimized Base Models** trained and evaluated on data scaled via `MinMaxScaler`.

In both runs, we will maintain the fixed parameters found in previous notebooks to evaluate which scaling strategy maximizes the overall **Recall**.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, accuracy_score, f1_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_Stacking")

2026/05/13 22:01:25 INFO mlflow.tracking.fluent: Experiment with name 'Classification_Stacking' does not exist. Creating a new experiment.


<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/EnsembleMethods/Stacking/StackingClassifier/mlruns/12'), creation_time=1778706085202, experiment_id='12', last_update_time=1778706085202, lifecycle_stage='active', name='Classification_Stacking', tags={}, workspace='default'>

In [2]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

# ---------------------------------------------------------
# DEFINING THE CHAMPION MODELS (Fixed winners)
# ---------------------------------------------------------

# KNN Champion
champion_knn = KNeighborsClassifier(
    n_neighbors=26, 
    weights='distance', 
    p=1
)

# Logistic Regression Champion
champion_lr = LogisticRegression(
    C=0.0011678781118622424, 
    penalty='l2', 
    solver='saga', 
    max_iter=2000
)

# Naive Bayes Champion
champion_nb = GaussianNB(
    var_smoothing=0.009541
)

# Decision Tree Champion (Previously identified)
champion_dt = DecisionTreeClassifier(
    criterion='entropy', 
    max_depth=29, 
    min_samples_leaf=3, 
    min_samples_split=7, 
    random_state=42
)

# List of base specialists for the Stack
champions_list = [
    ('knn', champion_knn),
    ('lr', champion_lr),
    ('nb', champion_nb),
    ('dt', champion_dt)
]

def log_metrics(y_true, y_pred, duration):
    """Log performance results to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# TOURNAMENT: 2 RUNS (Standardization vs Normalization)
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Stacking_{s_name}_Champions"):
        # Apply scaling to numerical features
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])

        # Initialize Stacking with fixed champions and default meta-model
        stack_model = StackingClassifier(
            estimators=champions_list, 
            final_estimator=LogisticRegression(),
            n_jobs=-1,
            cv=3 # Internal cross-validation for stacking
        )
        
        print(f"Starting training for {s_name} Stacking...")
        start_time = time.time()
        stack_model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        y_pred = stack_model.predict(X_test_scaled)
        
        # MLflow logging
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("meta_model", "LogisticRegression_Default")
        mlflow.log_param("optimization", "none_fixed_champions")
        
        # Log specific champion parameters for transparency
        mlflow.log_param("knn_k", 26)
        mlflow.log_param("lr_C", 0.001167)
        mlflow.log_param("nb_smoothing", 0.009541)
        mlflow.log_param("dt_depth", 29)
            
        log_metrics(y_test, y_pred, duration)
        print(f"Finished {s_name} in {duration:.2f} seconds.")

Starting training for Standardization Stacking...
Finished Standardization in 67.68 seconds.
Starting training for Normalization Stacking...
Finished Normalization in 67.11 seconds.


## Runs Summary

| Run | Scaler | KNN K | LR C | NB Smoothing | DT Depth | Accuracy | F1 | Recall | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Stacking_Standardization_Champions | Standardization | 26 | 0.001167 | 0.009541 | 29 | 0.89125 | 0.9072455115 | 0.8864166667 | 67.68s |
| Stacking_Normalization_Champions | Normalization | 26 | 0.001167 | 0.009541 | 29 | 0.88295 | 0.9013859051 | 0.8915833333 | 67.11s |

### Additional logged parameters
- `knn_k = 26`
- `lr_C = 0.001167`
- `nb_smoothing = 0.009541`
- `dt_depth = 29`
- `base_models = KNN, LogisticRegression, GaussianNB, DecisionTree (fixed champions)`

## Best Run Justification for Streamlit

For Streamlit, the decisive factor is not training time, but rather the quality of predictions when the user submits data: loading the model, applying preprocessing, and returning the classification. In this context, the best run is **Stacking_Standardization_Champions**.

The main reason is the overall balance of metrics. The version with **Standardization** presents the best **Accuracy** and best **F1**, while the version with **Normalization** has only a marginal gain in **Recall**. This gain in Recall does not offset the loss in Accuracy and F1, especially when both runs use exactly the same champion models and meta-model.

Final comparison:
- **Stacking_Standardization_Champions**: Accuracy 0.89125, F1 0.90725, Recall 0.88642
- **Stacking_Normalization_Champions**: Accuracy 0.88295, F1 0.90139, Recall 0.89158

For a medical application in Streamlit, the most balanced option is **Stacking_Standardization_Champions**, because it delivers the best overall performance and maintains a Recall very close to the best value. If the goal were to maximize only Recall, Normalization could be considered, but for more consistent and stable predictions, Standardization is the stronger choice.